# 05 Moment computations: mass, mean elevation, vertical spread

This notebook confirms the moment building blocks used both as a loss term and as a diagnostic. `PhysicsComponents.moment_sums` returns the zeroth, first, and second raw moments of a profile; the check derives mass, mean elevation, and vertical spread from them.

Modules exercised: `PhysicsComponents.moment_sums`, `PhysicsComponents.moments`, and the moment-agreement logic in `PhysicsQuantitiesCheck._moment_agreement`.

We compare the discretised moments against the analytic Gaussian moments and against an intentionally mismatched profile.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

torch.set_default_dtype(torch.float32)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "font.size"      : 11,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.titlesize" : 12,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

DEVICE = torch.device("cpu")


In [ ]:
from tools.data.gaussians                import GaussianMixture
from pipelines.backbone_pipeline.loss import PhysicsComponents

def to_check_tensor(profiles):
    t = torch.tensor(np.asarray(profiles, dtype=np.float32), device=DEVICE)
    return t.T.reshape(1, t.shape[1], 1, t.shape[0])

def gaussian_profile(x_axis, amp, mu, sigma):
    amps = np.asarray(amp,   dtype=np.float32).reshape(1, -1)
    mus  = np.asarray(mu,    dtype=np.float32).reshape(1, -1)
    sigs = np.asarray(sigma, dtype=np.float32).reshape(1, -1)
    return GaussianMixture.evaluate_batch(x_axis, amps, mus, sigs)[0]


In [ ]:
x_axis = np.linspace(-30.0, 50.0, 240).astype(np.float32)
dx     = float(x_axis[1] - x_axis[0])
xt     = torch.tensor(x_axis, device=DEVICE)

def moments_of(profile):
    t        = to_check_tensor(profile[None, :])
    s0, s1, s2 = PhysicsComponents.moment_sums(t, xt, dx)
    m0     = float(s0)
    mean   = float(s1 / s0)
    spread = float(torch.sqrt((s2 / s0 - (s1 / s0) ** 2).clamp(min=0.0)))
    return m0, mean, spread


## Discretised moments versus analytic Gaussian moments

For a Gaussian `amp * exp(-(x-mu)^2 / (2 sigma^2))` the analytic mass is `amp * sigma * sqrt(2 pi)`, the mean is `mu`, and the spread is `sigma`. We verify the discretised estimates recover these for a well-sampled scatterer.

In [ ]:
amp, mu, sigma = 1.3, 9.0, 5.0
profile        = gaussian_profile(x_axis, amp, mu, sigma)

m0, mean, spread = moments_of(profile)
analytic_mass    = amp * sigma * np.sqrt(2.0 * np.pi)

print(f"mass    discretised={m0:.4f}   analytic={analytic_mass:.4f}")
print(f"mean    discretised={mean:.4f}   analytic={mu:.4f}")
print(f"spread  discretised={spread:.4f}   analytic={sigma:.4f}")

## Moment estimates across a parameter sweep

We sweep the true mean and spread and confirm the recovered moments track the ground truth along the identity line.

In [ ]:
rng       = np.random.default_rng(SEED)
true_mu   = rng.uniform(-10.0, 25.0, size=40)
true_sig  = rng.uniform(2.0, 10.0,   size=40)
est_mu    = np.empty(40)
est_sig   = np.empty(40)

for i in range(40):
    p = gaussian_profile(x_axis, 1.0, float(true_mu[i]), float(true_sig[i]))
    _, est_mu[i], est_sig[i] = moments_of(p)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.5))
for ax, t, e, name in zip(axes, [true_mu, true_sig], [est_mu, est_sig], ["mean", "spread"]):
    lo, hi = float(min(t.min(), e.min())), float(max(t.max(), e.max()))
    ax.plot([lo, hi], [lo, hi], color="0.4", ls="--", lw=0.9)
    ax.scatter(t, e, s=18, color="C0")
    ax.set_xlabel(f"true {name}")
    ax.set_ylabel(f"estimated {name}")
    ax.set_title(f"{name} recovery")
fig.tight_layout()
plt.show()

## Moment loss separates matched from mismatched profiles

`PhysicsComponents.moments` aggregates normalised mass, mean, and spread errors. We show it is near zero when a profile is compared to itself and grows when the mean or spread is perturbed.

In [ ]:
base    = gaussian_profile(x_axis, 1.0, 8.0, 4.0)
shifted = gaussian_profile(x_axis, 1.0, 16.0, 4.0)
widened = gaussian_profile(x_axis, 1.0, 8.0, 9.0)

bt = to_check_tensor(base[None, :])
def m_loss(pred):
    pt = to_check_tensor(pred[None, :])
    return float(PhysicsComponents.moments(pt, bt, xt, dx, 1e-3, (1.0, 1.0, 1.0)))

print(f"moments(base,  base ) = {m_loss(base):.5f}")
print(f"moments(shift, base ) = {m_loss(shifted):.5f}")
print(f"moments(wide,  base ) = {m_loss(widened):.5f}")

## Expected outcome

The discretised moments reproduce the analytic Gaussian moments to within sampling error, the recovery scatter falls on the identity line, and the moment loss is near zero for a self-comparison while rising clearly under a mean shift or a spread change, confirming the term's discriminative power.